In [2]:
from utilities import orthogonal_projection, parallel_projection, oblique_projection, randdir

M = 6 # number of mics
K = 3 # number of sources

G = randdir(M, K) # random steering vectors (old sources)
h = randdir(M, 1) # random steering vector (new source)
Ppa_G = parallel_projection(G)
Por_G = orthogonal_projection(G)
Pob_G = oblique_projection(G, h)




In [4]:
import torch
import matplotlib.pyplot as plt

# Try to determine dtype/device from G, fallback to defaults
dtype = G.dtype if isinstance(G, torch.Tensor) else torch.float32
device = G.device if isinstance(G, torch.Tensor) else torch.device('cpu')

# 1. Store proper projections as the baseline
Ppa_proper = parallel_projection(G)
Por_proper = orthogonal_projection(G)
Pob_proper = oblique_projection(G, h)

norm_Ppa = torch.linalg.norm(Ppa_proper)
norm_Por = torch.linalg.norm(Por_proper)
norm_Pob = torch.linalg.norm(Pob_proper)

# Metrics storage
n_zeros_list = list(range(1, 2 * K + 1))
errors_rel_norm = {'pa': [], 'or': [], 'ob': []}
errors_max_elem = {'pa': [], 'or': [], 'ob': []}

print(f"{'Zeros':<6} | {'Status':<10} | {'Max Elem Err (Pa)':<18} | {'Rel Norm Err (Pa)':<18}")
print("-" * 65)

for n in n_zeros_list:
    # Append n zero columns to G
    zeros = torch.zeros((M, n), dtype=dtype, device=device)
    G_z = torch.cat([G, zeros], dim=1)
    
    try:
        # Calculate new projections
        Ppa_z = parallel_projection(G_z)
        Por_z = orthogonal_projection(G_z)
        Pob_z = oblique_projection(G_z, h)
        
        # Calculate Rel Norm Error ( ||P_z - P_proper|| / ||P_proper|| )
        rel_norm_pa = torch.linalg.norm(Ppa_z - Ppa_proper) / norm_Ppa
        rel_norm_or = torch.linalg.norm(Por_z - Por_proper) / norm_Por
        rel_norm_ob = torch.linalg.norm(Pob_z - Pob_proper) / norm_Pob
        
        # Calculate Max Element-wise Error
        max_elem_pa = torch.max(torch.abs(Ppa_z - Ppa_proper))
        max_elem_or = torch.max(torch.abs(Por_z - Por_proper))
        max_elem_ob = torch.max(torch.abs(Pob_z - Pob_proper))
        
        # Store
        errors_rel_norm['pa'].append(rel_norm_pa.item())
        errors_rel_norm['or'].append(rel_norm_or.item())
        errors_rel_norm['ob'].append(rel_norm_ob.item())
        
        errors_max_elem['pa'].append(max_elem_pa.item())
        errors_max_elem['or'].append(max_elem_or.item())
        errors_max_elem['ob'].append(max_elem_ob.item())
        
        print(f"{n:<6} | {'Success':<10} | {max_elem_pa.item():<18.2e} | {rel_norm_pa.item():<18.2e}")
        
    except Exception as e:
        print(f"{n:<6} | FAILED     | {str(e)[:100]}")
        for key in ['pa', 'or', 'ob']:
            errors_rel_norm[key].append(float('nan'))
            errors_max_elem[key].append(float('nan'))

# --- PLOTTING ---
# Only plot if we had at least one success
if (~torch.isnan(torch.tensor(errors_rel_norm['pa']))).any():
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Plot 1: Relative Norm Errors
    ax1.plot(n_zeros_list, errors_rel_norm['pa'], 'o-', label='Parallel')
    ax1.plot(n_zeros_list, errors_rel_norm['or'], 's-', label='Orthogonal')
    ax1.plot(n_zeros_list, errors_rel_norm['ob'], '^-', label='Oblique')
    ax1.set_yscale('log')
    ax1.set_title("Relative Norm Error vs Zero Columns")
    ax1.set_xlabel("Number of appended zero columns")
    ax1.set_ylabel(r"$|| P_{zeros} - P_{proper} ||_F \; / \; || P_{proper} ||_F$")
    ax1.grid(True, which="both", ls="--", alpha=0.5)
    ax1.legend()

    # Plot 2: Max Element-wise Errors
    ax2.plot(n_zeros_list, errors_max_elem['pa'], 'o-', label='Parallel')
    ax2.plot(n_zeros_list, errors_max_elem['or'], 's-', label='Orthogonal')
    ax2.plot(n_zeros_list, errors_max_elem['ob'], '^-', label='Oblique')
    ax2.set_yscale('log')
    ax2.set_title("Maximum Element-wise Error vs Zero Columns")
    ax2.set_xlabel("Number of appended zero columns")
    ax2.set_ylabel(r"$\max | P_{zeros} - P_{proper} |$")
    ax2.grid(True, which="both", ls="--", alpha=0.5)
    ax2.legend()

    plt.tight_layout()
    plt.show()
else:
    print("All attempts failed. Your projection operators likely use standard inverse/solve instead of pinv.")

Zeros  | Status     | Max Elem Err (Pa)  | Rel Norm Err (Pa) 
-----------------------------------------------------------------
1      | FAILED     | torch.linalg.solve: The solver failed because the input matrix is singular.
2      | FAILED     | torch.linalg.solve: The solver failed because the input matrix is singular.
3      | FAILED     | torch.linalg.solve: The solver failed because the input matrix is singular.
4      | FAILED     | torch.linalg.solve: The solver failed because the input matrix is singular.
5      | FAILED     | torch.linalg.solve: The solver failed because the input matrix is singular.
6      | FAILED     | torch.linalg.solve: The solver failed because the input matrix is singular.
All attempts failed. Your projection operators likely use standard inverse/solve instead of pinv.


In [4]:
G0 = G[:, 0:0] # Empty matrix with shape (M, 0)
Ppa_G0 = parallel_projection(G0)
Por_G0 = orthogonal_projection(G0)
Pob_G0 = oblique_projection(G0, h)

In [7]:
Pob_G0

tensor([[0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j],
        [0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j],
        [0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j],
        [0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j],
        [0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j],
        [0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j]], device='cuda:0',
       dtype=torch.complex128)